In [1]:
# Colab Setup
import sys
print(f"Python version: {sys.version}")
!nvidia-smi

Python version: 3.12.12 (main, Oct 10 2025, 08:52:57) [GCC 11.4.0]
Thu Feb 19 17:10:12 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   62C    P8             10W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |     

In [22]:
import os
import json
import time
import random
import pandas as pd
import torch
from datasets import Dataset
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    TrainingArguments,
    Trainer,
    DataCollatorWithPadding,
    set_seed,
)
from sklearn.metrics import (
    f1_score,
    precision_recall_fscore_support,
    roc_auc_score,
    average_precision_score,
)

In [3]:
print("Data structure:")

try:
    from google.colab import drive
    drive.mount('/content/drive', force_remount=False)

    train_path = "/content/drive/MyDrive/submission_data/train_data.csv"
    if os.path.exists(train_path):
        train_df = pd.read_csv(train_path)
        print(f"\n Train data loaded ({len(train_df)} samples)")
        print("Columns available:")
        for col in train_df.columns[:7]:
            print(f"  - {col}")

        print("\nFirst few rows:")
        print(train_df[['text', 'domain'] + [c for c in train_df.columns if c in ['anxiety', 'anger', 'sadness']]].head())
    else:
        print(f" Train data not found at: {train_path}")

except Exception as e:
    print(f"Error: {e}")

Data structure:
Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).

 Train data loaded (24169 samples)
Columns available:
  - text
  - dataset_name
  - domain
  - labelanx
  - labelanger
  - labelsad
  - row_id

First few rows:
                                                text           domain  anger
0  When Iraq invaded Kuwait and was posed to inva...           crisis   1.89
1  @Ukraine_DAO need help relocating. Anyone out ...           crisis   0.00
2  Overwhelmed by all the orders we�ve received s...  social_movement   0.00
3  RT @Chardwa13: Israel, knowing that a Palestin...           crisis   0.00
4  RT @Write7Dave: Russian troops are “becoming i...           crisis   5.00


In [18]:
import numpy as np
import os
import json
import time
import random
import pandas as pd
import torch
from datasets import Dataset
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    TrainingArguments,
    Trainer,
    DataCollatorWithPadding,
    set_seed,
)
from sklearn.metrics import (
    f1_score,
    precision_recall_fscore_support,
    roc_auc_score,
    average_precision_score,
)
from dataclasses import dataclass
from typing import Dict, List, Tuple

@dataclass
class Config:
    data_dir: str = "/content/drive/MyDrive/submission_data"
    train_file: str = "/content/drive/MyDrive/submission_data/train_data.csv"
    val_file: str = "/content/drive/MyDrive/submission_data/val_data.csv"
    test_file: str = "/content/drive/MyDrive/submission_data/test_data.csv"

    model_name: str = "roberta-base"
    max_length: int = 256

    # Fix: Updated label names to match actual column names in the CSV
    labels: Tuple[str, str, str] = ("labelanx", "labelanger", "labelsad")
    text_col: str = "text"

    id_col: str = "row_id"
    keep_cols: Tuple[str, ...] = ("row_id", "domain", "dataset_name")

    out_dir: str = "/content/drive/MyDrive/submission/roberta_final"
    seed: int = 42

    epochs: int = 3
    lr: float = 2e-5
    weight_decay: float = 0.01
    warmup_ratio: float = 0.1
    train_bs: int = 16
    eval_bs: int = 32

    do_threshold_tuning: bool = True
    threshold_grid: Tuple[float, float, float] = (0.10, 0.70, 0.01)


CFG = Config()

def set_all_seeds(seed: int) -> None:
    set_seed(seed)
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

# converts logits t probabilities
def sigmoid(x: np.ndarray) -> np.ndarray:
    x = np.clip(x, -30, 30)
    return 1.0 / (1.0 + np.exp(-x))

def safe_auc_and_ap(y_true: np.ndarray, y_score: np.ndarray) -> Tuple[float, float]:
    if len(np.unique(y_true)) < 2:
        return float("nan"), float("nan")
    return float(roc_auc_score(y_true, y_score)), float(average_precision_score(y_true, y_score))

# converts pandas df to hugging face
def to_hf_dataset(df: pd.DataFrame, text_col: str, label_cols: List[str], keep_cols: List[str]) -> Dataset:
    labels = df[label_cols].values.astype(np.float32).tolist()

    cols = [text_col] + keep_cols
    cols = [c for c in cols if c in df.columns]

    ds = Dataset.from_pandas(df[cols], preserve_index=False)
    if text_col != "text":
        ds = ds.rename_column(text_col, "text")

    ds = ds.add_column("labels", labels)
    return ds

def load_split(path: str, text_col: str, label_cols: List[str], id_col: str) -> pd.DataFrame:
    df = pd.read_csv(path)
    df = df.loc[:, ~df.columns.astype(str).str.startswith("Unnamed:")].copy()

    if text_col not in df.columns:
        raise ValueError(f"Missing text column '{text_col}'")

    df[text_col] = df[text_col].astype(str).fillna("").str.strip()
    df = df[df[text_col].str.len() > 0].copy()

    for c in label_cols:
        df[c] = pd.to_numeric(df[c], errors="coerce").fillna(0).astype(int).clip(0, 1)

    df[id_col] = df[id_col].astype(str)

    return df.reset_index(drop=True)

def metrics_from_probs(
    y_true: np.ndarray,
    y_prob: np.ndarray,
    threshold: np.ndarray,
    label_names: List[str],
    prefix:str,
) -> Dict[str, float]:
    thresholds = threshold.astype(np.float32).reshape(1, -1)
    y_pred = (y_prob >= thresholds).astype(int)

    out: Dict[str, float] = {}
    out[f"{prefix}_macro_f1"] = f1_score(y_true, y_pred, average="macro")
    out[f"{prefix}_micro_f1"] = f1_score(y_true, y_pred, average="micro")

    aucs = []
    for i, name in enumerate(label_names):
      p, r, f, _ = precision_recall_fscore_support(
          y_true[:, i], y_pred[:, i], average="binary", zero_division=0
      )
      out[f"{prefix}_{name}_precision"] = p
      out[f"{prefix}_{name}_recall"] = r
      out[f"{prefix}_{name}_f1"] = f

      auc,ap = safe_auc_and_ap(y_true[:,i], y_prob[:,i])
      out[f"{prefix}_{name}_auc"] = auc
      out[f"{prefix}_{name}_ap"] = ap
      if not np.isnan(auc):
        aucs.append(auc)
    out["auc_mean"] = np.mean(aucs) if aucs else float("nan")
    return out

def tune_threshold_f1(
    y_true: np.ndarray,
    y_prob: np.ndarray,
    label_names: List[str],
    grid: Tuple[float, float, float],
) -> Tuple[np.ndarray,Dict[str, float]]:
    grid = np.arange(grid[0], grid[1], grid[2])
    best_t = np.full((y_true.shape[1],), 0.5, dtype=np.float32)
    best_f1 = np.full((y_true.shape[1],), -1.0, dtype=np.float32)

    for i in range(y_true.shape[1]):
      yt = y_true[:,i].astype(int)
      yp = y_prob[:,i]
      for t in grid:
          pred = (yp >= t).astype(int)
          f1 = f1_score(yt, pred, average="binary", zero_division = 0)
          if f1 > best_f1[i]:
            best_f1[i] = f1
            best_t[i] = t

    info = {f"best_threshold_{label_names[i]}": float(best_t[i]) for i in range(len(label_names))}
    info.update({f"best_f1_{label_names[i]}": float(best_f1[i]) for i in range(len(label_names))})
    info["grid_start"] = grid[0]
    info["grid_end"] = grid[1]
    info["grid_step"] = grid[2]
    return best_t, info

def compute_metrics(eval_pred):
    logits = eval_pred.predictions if hasattr(eval_pred, "predictions") else eval_pred[0]
    labels = eval_pred.label_ids  if hasattr(eval_pred, "label_ids") else eval_pred[1]
    probs = sigmoid(logits)
    y_true = labels.astype(int)
    y_prob = probs
    y_pred = (probs >= 0.5).astype(int)

    out = {
        "f1_macro": f1_score(y_true, y_pred, average="macro", zero_division= 0),
        "f1_micro": f1_score(y_true, y_pred, average="micro", zero_division= 0),
    }

    aucs = []
    for i,name in enumerate(CFG.labels):
      auc, _ = safe_auc_and_ap(y_true[:,i], y_prob[:,i])
      out [f"{name}_auc"] = auc
      if not np.isnan(auc):
        aucs.append(auc)
    out ["auc_mean"] = np.mean(aucs) if aucs else float("nan")
    return out

def save_pred_csv(
    df_orig: pd.DataFrame,
    y_prob: np.ndarray,
    label_names: List[str],
    label_cols: List[str],
    id_col: str,
    out_dir: str,
    keep_cols: List[str],
    prefix: str,
    threshold: np.ndarray,
) -> None:

    thresholds_for_save = threshold.astype(np.float32).reshape(1, -1)
    pred = (y_prob >= thresholds_for_save).astype(int)

    out = df_orig[keep_cols].copy()
    for i, col in enumerate(label_cols):
        out[f"{prefix}{col}_pred"] = pred[:, i].astype(int)
        out[f"{prefix}{col}_prob"] = y_prob[:, i].astype(np.float32)

    out.to_csv(os.path.join(out_dir, f"{prefix}submission.csv"), index=False)

In [25]:
def main() -> None:
    set_all_seeds(CFG.seed)
    os.makedirs(CFG.out_dir, exist_ok=True)

    label_cols = list(CFG.labels)
    keep_cols = list(CFG.keep_cols)

    train_path = os.path.join(CFG.data_dir, CFG.train_file)
    val_path = os.path.join(CFG.data_dir, CFG.val_file)
    test_path = os.path.join(CFG.data_dir, CFG.test_file)

    print("Loading data")
    train_df = load_split(train_path, CFG.text_col, label_cols, CFG.id_col)
    val_df = load_split(val_path, CFG.text_col, label_cols, CFG.id_col)
    test_df = load_split(test_path, CFG.text_col, label_cols, CFG.id_col)

    print(f"Train: {len(train_df)} Val: {len(val_df)} Test: {len(test_df)}")
    print("Train class distribution:", train_df[label_cols].mean().to_dict())
    print("Val class distribution:", val_df[label_cols].mean().to_dict())
    print("Test class distribution:", test_df[label_cols].mean().to_dict())

    train_ds = to_hf_dataset(train_df, CFG.text_col, label_cols, keep_cols)
    val_ds = to_hf_dataset(val_df, CFG.text_col, label_cols, keep_cols)
    test_ds = to_hf_dataset(test_df, CFG.text_col, label_cols, keep_cols)

    tokenizer = AutoTokenizer.from_pretrained(CFG.model_name)

    def tokenize(batch):
        return tokenizer(batch["text"], padding=False, truncation=True, max_length=CFG.max_length)

    train_ds = train_ds.map(tokenize, batched=True)
    val_ds = val_ds.map(tokenize, batched=True)
    test_ds = test_ds.map(tokenize, batched=True)

    data_collator = DataCollatorWithPadding(tokenizer=tokenizer)

    model = AutoModelForSequenceClassification.from_pretrained(
        CFG.model_name,
        num_labels=len(label_cols),
        problem_type="multi_label_classification",
    )

    training_args = TrainingArguments(
        output_dir=CFG.out_dir,
        num_train_epochs=CFG.epochs,
        learning_rate=CFG.lr,
        weight_decay=CFG.weight_decay,
        warmup_ratio=CFG.warmup_ratio,
        per_device_train_batch_size=CFG.train_bs,
        per_device_eval_batch_size=CFG.eval_bs,
        eval_strategy="epoch",
        save_strategy="epoch",
        save_total_limit=2,
        load_best_model_at_end=True,
        metric_for_best_model="auc_mean",
        greater_is_better=True,
        fp16=torch.cuda.is_available(),
        logging_steps=100,
        report_to="none",
    )

    trainer = Trainer(
        model=model,
        args=training_args,
        train_dataset=train_ds,
        eval_dataset=val_ds,
        data_collator=data_collator,
        compute_metrics=compute_metrics,
    )

    resume_checkpoint = None

    if os.path.exists(os.path.join(CFG.out_dir, "trainer_state.json")):
        print(f"Found existing trainer_state.json in {CFG.out_dir}. Checking for latest checkpoint to resume.")

        # Manually find the latest checkpoint directory
        checkpoints = [d for d in os.listdir(CFG.out_dir) if d.startswith('checkpoint-') and os.path.isdir(os.path.join(CFG.out_dir, d))]
        if checkpoints:
            # Sort checkpoints by step number (e.g., checkpoint-100, checkpoint-200)
            latest_checkpoint = sorted(checkpoints, key=lambda x: int(x.split('-')[1]))[-1]
            resume_checkpoint = os.path.join(CFG.out_dir, latest_checkpoint)
            print(f"Resuming from detected checkpoint: {resume_checkpoint}")
        else:
            print(f"trainer_state.json found but no valid checkpoint directory in {CFG.out_dir}. Starting from scratch.")
    else:
        print("No existing trainer_state.json found. Starting training from scratch.")

    print("Training...")
    t0 = time.time()
    trainer.train(resume_from_checkpoint=resume_checkpoint)
    print("Best checkpoint:", trainer.state.best_model_checkpoint)
    train_time = time.time() - t0
    print(f"Training time: {train_time:.2f}s")

    print("Saving best model")
    trainer.save_model(os.path.join(CFG.out_dir, "best_model"))
    print("Saving tokenizer")
    tokenizer.save_pretrained(os.path.join(CFG.out_dir, "tokenizer"))
    print("Saving trainer state for potential resume")
    trainer.save_state()

    # post-hoc eval, threshold 0.5, val-tuned
    default_t = np.full((len(label_cols),), 0.5, dtype=np.float32)

    print("Predicting on Val")
    val_pred = trainer.predict(val_ds)
    val_y_true = np.array(val_pred.label_ids).astype(int)
    val_y_prob = sigmoid(val_pred.predictions)

    val_metrics_default = metrics_from_probs(val_y_true, val_y_prob, default_t, label_cols, "val")

    if CFG.do_threshold_tuning:
        best_t, tune_info = tune_threshold_f1(val_y_true, val_y_prob, label_cols, CFG.threshold_grid)
        val_metrics_tuned = metrics_from_probs(val_y_true, val_y_prob, best_t, label_cols, "val_tuned")
    else:
        best_t = default_t.copy()
        tune_info = {"disabled": True}
        val_metrics_tuned = metrics_from_probs(val_y_true, val_y_prob, best_t, label_cols, "val_tuned")

    print("Predicting on Test")
    test_pred = trainer.predict(test_ds)
    test_y_prob = sigmoid(test_pred.predictions)
    test_y_true = np.array(test_pred.label_ids).astype(int)

    test_metrics_default = metrics_from_probs(test_y_true, test_y_prob, default_t, label_cols, "test")
    test_metrics_tuned = metrics_from_probs(test_y_true, test_y_prob, best_t, label_cols, "test_tuned")

    results = {
        "config": {**CFG.__dict__, "label_cols": label_cols},
        "dataset_stats": {
            "n_train": int(len(train_df)),
            "n_val": int(len(val_df)),
            "n_test": int(len(test_df)),
            "train_prevalence": {k: float(v) for k, v in train_df[label_cols].mean().to_dict().items()},
            "val_prevalence": {k: float(v) for k, v in val_df[label_cols].mean().to_dict().items()},
            "test_prevalence": {k: float(v) for k, v in test_df[label_cols].mean().to_dict().items()},
        },
        "train_time": float(train_time),
        "threshold_tuning": tune_info,
        "val_metrics_default_0.5": val_metrics_default,
        "val_metrics_tuned": val_metrics_tuned,
        "test_metrics_default_0.5": test_metrics_default,
        "test_metrics_tuned": test_metrics_tuned,
        "best_thresholds_used_for_diagnostic": {label_cols[i]: float(best_t[i]) for i in range(len(label_cols))},
    }

    out_json = os.path.join(CFG.out_dir, "results_roberta.json")
    with open(out_json, "w", encoding="utf-8") as f:
        json.dump(results, f, indent=2)

    keep_cols_existing = [c for c in keep_cols if c in val_df.columns]
    label_cols_to_keep = [c for c in label_cols if col in test_df.columns]
    keep_cols_existing.extend(label_cols_to_keep)

    save_pred_csv(
        df_orig=val_df,
        y_prob=val_y_prob,
        label_names=label_cols,
        label_cols=label_cols,
        id_col=CFG.id_col,
        out_dir=CFG.out_dir,
        keep_cols=keep_cols_existing,
        prefix="val_roberta_",
        threshold=default_t
    )

    save_pred_csv(
        df_orig=test_df,
        y_prob=test_y_prob,
        label_names=label_cols,
        label_cols=label_cols,
        id_col=CFG.id_col,
        out_dir=CFG.out_dir,
        keep_cols=keep_cols_existing,
        prefix="test_roberta_",
        threshold=default_t
    )

    save_pred_csv(
        df_orig=val_df,
        y_prob=val_y_prob,
        label_names=label_cols,
        label_cols=label_cols,
        id_col=CFG.id_col,
        out_dir=CFG.out_dir,
        keep_cols=keep_cols_existing,
        prefix="val_roberta_tuned_",
        threshold=best_t
    )

    save_pred_csv(
        df_orig=test_df,
        y_prob=test_y_prob,
        label_names=label_cols,
        label_cols=label_cols,
        id_col=CFG.id_col,
        out_dir=CFG.out_dir,
        keep_cols=keep_cols_existing,
        prefix="test_roberta_tuned_",
        threshold=best_t
    )

    print("\nDone.")
    print("Saved model to:", CFG.out_dir)
    print("Saved results to:", out_json)
    print("Saved prediction CSVs to:", CFG.out_dir)

    print("\nDIAGNOSTIC thresholds (if tuning disabled, these will be 0.5):")
    print({label_cols[i]: float(best_t[i]) for i in range(len(label_cols))})

    print("\nVAL f1_micro 0.5:", results["val_metrics_default_0.5"]["val_micro_f1"])
    print("VAL f1_micro val-tuned:", results["val_metrics_tuned"]["val_tuned_micro_f1"])

    print("\nTEST f1_micro 0.5:", results["test_metrics_default_0.5"]["test_micro_f1"])
    print("TEST f1_micro val-tuned:", results["test_metrics_tuned"]["test_tuned_micro_f1"])

    print("\nTEST AUC/AP (threshold-free, per label) 0.5 block:")
    print({k: v for k, v in results["test_metrics_default_0.5"].items() if k.endswith("_auc") or k.endswith("_ap")})


if __name__ == "__main__":
    main()

Loading data
Train: 24169 Val: 6000 Test: 30008
Train class distribution: {'labelanx': 0.07488932103107286, 'labelanger': 0.308121974430055, 'labelsad': 0.07071041416690803}
Val class distribution: {'labelanx': 0.07483333333333334, 'labelanger': 0.2995, 'labelsad': 0.08166666666666667}
Test class distribution: {'labelanx': 0.20474540122633964, 'labelanger': 0.3054518794988003, 'labelsad': 0.10370567848573714}


Map:   0%|          | 0/24169 [00:00<?, ? examples/s]

Map:   0%|          | 0/6000 [00:00<?, ? examples/s]

Map:   0%|          | 0/30008 [00:00<?, ? examples/s]

Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

...

Best checkpoint: /content/drive/MyDrive/submission/roberta_final/checkpoint-4533
Training time: 8.27s
Saving best model


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Saving tokenizer
Saving trainer state for potential resume
Predicting on Val


Predicting on Test



Done.
Saved model to: /content/drive/MyDrive/submission/roberta_final
Saved results to: /content/drive/MyDrive/submission/roberta_final/results_roberta.json
Saved prediction CSVs to: /content/drive/MyDrive/submission/roberta_final

DIAGNOSTIC thresholds (if tuning disabled, these will be 0.5):
{'labelanx': 0.6800000071525574, 'labelanger': 0.6600000262260437, 'labelsad': 0.5899999737739563}

VAL f1_micro 0.5: 0.9676945668135095
VAL f1_micro val-tuned: 0.9701767304860088

TEST f1_micro 0.5: 0.6593748918348274
TEST f1_micro val-tuned: 0.6597766525629465

TEST AUC/AP (threshold-free, per label) 0.5 block:
{'test_labelanx_auc': 0.6787112000828044, 'test_labelanx_ap': 0.5886835935500216, 'test_labelanger_auc': 0.7879412140547786, 'test_labelanger_ap': 0.7160198430813869, 'test_labelsad_auc': 0.7664178401543639, 'test_labelsad_ap': 0.5749409498563662}


In [24]:
import os

output_dir = CFG.out_dir

print(f"Contents of output directory: {output_dir}")
if os.path.exists(output_dir):
    for item in os.listdir(output_dir):
        full_path = os.path.join(output_dir, item)
        if os.path.isdir(full_path):
            print(f"  [DIR] {item}")
        else:
            print(f"  [FILE] {item}")
else:
    print(f"Output directory does not exist: {output_dir}")

Contents of output directory: /content/drive/MyDrive/submission/roberta_final
  [DIR] best_model
  [DIR] tokenizer
  [FILE] roberta_submission.csv
  [FILE] roberta_tuned_submission.csv
  [FILE] val_roberta_submission.csv
  [FILE] test_roberta_submission.csv
  [FILE] val_roberta_tuned_submission.csv
  [FILE] test_roberta_tuned_submission.csv
  [FILE] results_roberta.json
  [DIR] checkpoint-3022
  [DIR] checkpoint-4533
  [FILE] trainer_state.json


In [6]:
# Analyze results with confusion matrices and detailed metrics